In [1]:
from bertopic import BERTopic
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def bertopic_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

In [ ]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    bertopic_analysis(nurse_notes[key])
    all_texts.extend(nurse_notes[key])

-----------P1-----------
Number of texts: 603


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6752423520017569
Diversity: 0.5375
Inverse Redundancy: 0.9235507246376812
Time (seconds): 37.977317094802856
----- Cluster Topics -----
['plan', 'of', 'care', 'to', 'and', 'the', 'as', 'resident', 'no', 'all']
['rollator', 'mobilising', 'with', 'good', 'given', 'doctor', 'form', 'steroids', 'charted', 'complaints']
['relaxed', 'by', 'staff', 'adl', 'her', 'content', 'taken', 'and', 'appears', 'with']
['checks', 'safety', 'on', 'continued', 'comfortable', 'meds', 'due', 'taken', 'settled', 'asleep']
['settled', 'bed', 'sleeping', 'post', 'on', 'medications', 'to', 'independently', 'be', 'checks']
['toiletting', 'ongoing', 'asleep', 'self', 'comfortable', 'checks', 'resident', 'toileting', 'peacefully', 'required']
['mobile', 'restaurant', 'usual', 'attended', 'her', 'independent', 'form', 'needs', 'in', 'due']
['well', 'remains', 'report', 'sleeping', 'caring', 'pleasant', 'and', 'no', 'pleasantly', 'continues']
['peaceful', 'toiletting', 'ongoing', 'asleep', 'self', 'check

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
python(9961) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9965) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9966) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9968) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9969) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9970) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(9971) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Coherence: 0.7343058823795673
Diversity: 0.5833333333333334
Inverse Redundancy: 0.9304347826086956
Time (seconds): 374.825532913208
----- Cluster Topics -----
['resident', 'needs', 'and', 'as', 'in', 'due', 'checks', 'continued', 'care', 'assisted']
['to', 'sleeping', 'bed', 'settled', 'kept', 'well', 'on', 'checks', 'resident', 'comfortable']
['wound', '2nd', 'right', 'dressing', 'plan', '1st', 'digit', 'developed', 'left', 'has']
['inhalers', 'voiced', 'complaints', 'good', 'nil', 'appears', 'form', 'attended', 'charted', 'given']
['doctor', 'chesty', 'infection', 'antibiotic', 'of', 'chest', 'commenced', 'tract', 'respiratory', 'plan']
['peaceful', 'ongoing', 'asleep', 'skin', 'care', 'continued', 'checks', 'needs', 'assisted', 'resident']
['eye', 'drops', 'settled', 'kept', 'post', 'all', 'maintained', 'medications', 'needs', 'sleeping']
['adls', 'breakfast', 'dinning', 'intake', 'unit', 'chatty', 'planned', 'area', 'reported', 'with']
['adl', 'her', 'taken', 'morning', 'this', 'ch

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
bertopic_analysis(all_texts)

Number of texts: 12225


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.5921785060833745
Diversity: 0.3062314540059347
Inverse Redundancy: 0.9662091988130563
Time (seconds): 18.074954986572266
----- Cluster Topics -----
['gp', 'note', 'doctor', 'receive', 'pain', 'sleep', 'till', 'keep', 'med', 'morning']
['inhaler', 'club', 'social', 'nebs', 'aspiration', 'antibiotic', 'laxose', 'listen', 'music', 'knitting']
['wheel', 'observe', 'intake', 'today', 'chair', 'pu', 'note', 'aid', 'skin', 'medicine']
['complaint', 'voice', 'slt', 'medication', 'attend', 'adhere', 'rolator', 'nil', 'take', 'activity']
['alert', 'medicine', 'food', 'bright', 'fluid', 'render', 'intake', 'today', 'intact', 'pu']
['actively', 'integrity', 'general', 'supervised', 'main', 'progress', 'insitu', 'post', 'randomly', 'toilet']
['compliant', 'charted', 'meds', 'assisted', 'adls', 'maintain', 'night', 'settle', 'safety', 'versatili']
['diet', 'weight', 'nutritional', 'kg', 'dietetic', 'dietitian', 'protein', 'loss', 'vitamin', 'year']
['mobile', 'restaurant', 'independent'